# Análisis de Video — Pipeline de Tracking (F1)

Procesa el video de un partido y lo convierte en **tracking en coordenadas OPTA 0-100**
(el mismo sistema que Sofascore — todo se cruza directo).

**Flujo:** video → detección (YOLOv8) → tracking (ByteTrack) → equipos (clusters de
camiseta) → homografía (píxeles → cancha) → `tracking/positions.csv` → **validación
contra Sofascore**.

**Convención:** el video va en `data/temporadas/{año}/partidos/{match_key}/video/`
(cualquier formato: mp4, mkv, avi, mov); los CSV de Sofascore de ese partido van en
`.../{match_key}/sofascore/`.

> Corré esta celda solo si tenés GPU CUDA local con el stack de
> `requirements-video.txt` instalado. Si no, procesá en Colab con
> `procesar_video_colab.ipynb` y usá este notebook solo para las secciones 3-4
> (validación y vista) sobre el `tracking/` que copiaste de vuelta.

In [ ]:
import os, sys

# Recarga automática de módulos src/ al editarlos (sin reiniciar kernel)
%load_ext autoreload
%autoreload 2

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import config
from src.video import (
    check_environment, environment_ready, process_video,
    load_tracking, track_summary, players_per_frame,
    compare_with_lineups, positions_plausibility,
)

# ============================================================
# CONFIGURAR ACÁ
SEASON        = '2026'
MATCH_KEY     = '2026-09-06_vs_estudiantes_caseros_h'  # partido a procesar
START_SECONDS = 0      # TODO: segundo del ARCHIVO donde arranca el partido (saque inicial)
TEST_SECONDS  = 900    # F1: primeros 15 min DESDE START_SECONDS; None = video completo
# ============================================================

VIDEO_DIR     = config.match_video_dir(MATCH_KEY, SEASON)
SOFASCORE_DIR = config.match_sofascore_dir(MATCH_KEY, SEASON)
TRACKING_DIR  = config.match_tracking_dir(MATCH_KEY, SEASON)

print(f'Partido    : {MATCH_KEY}')
print(f'Video en   : {VIDEO_DIR}')
print(f'Sofascore  : {SOFASCORE_DIR}')
print(f'Tracking   : {TRACKING_DIR}')

## 1. Checklist del entorno

Antes de procesar hay que tener: dependencias instaladas, modelos accesibles (API key
de Roboflow, backend por defecto) y el video del partido.

**Instalación (una sola vez, en una terminal con el venv activado):**
```bash
pip install -r requirements.txt
pip install -r requirements-video.txt   # solo si corrés el pipeline localmente
```

Este chequeo suele salir 🔴 en la PC local (Python 3.13, sin `inference`) — es
esperado. Para procesar de verdad, usar `procesar_video_colab.ipynb` y volver acá
solo para las secciones 3-4.

In [ ]:
checks = check_environment(MATCH_KEY, SEASON)
for name, (ok, detail) in checks.items():
    print(f"{'✅' if ok else '❌'} {name}: {detail}")

LISTO = environment_ready(checks)
print(f"\n{'🟢 Entorno listo para procesar acá.' if LISTO else '🔴 Falta algo para procesar en esta PC (¿usás Colab?).'}")

## 2. Procesar el video (solo si el entorno local está listo)

Si no está listo, se intenta cargar un `tracking/` ya generado en Colab y copiado acá.

In [ ]:
if LISTO:
    df_pos, df_ball, meta = process_video(
        MATCH_KEY, SEASON,
        start_seconds=START_SECONDS,
        max_seconds=TEST_SECONDS,   # None = video completo
        save=True,
    )
    print(f"\nPosiciones: {len(df_pos)} filas | Pelota: {len(df_ball)} filas")
else:
    tr = load_tracking(MATCH_KEY, SEASON)
    df_pos, df_ball, meta = tr['positions'], tr['ball'], tr['meta'] or {}
    if df_pos is not None:
        print(f"Tracking cargado (de Colab): {len(df_pos)} filas ({meta.get('fecha','?')})")
    else:
        print('Sin entorno listo y sin tracking previo — procesá en procesar_video_colab.ipynb '
              'y copiá tracking/ acá.')

## 3. Validación contra Sofascore

Sofascore (`sofascore/lineups_clean.csv` de este partido) es la verdad oficial: si el
tracking se aleja mucho en jugadores/minutos, o la homografía pone puntos fuera de la
cancha, la extracción tiene problemas y NO hay que usarla para análisis todavía.

In [ ]:
if df_pos is not None and not df_pos.empty:
    # Plausibilidad espacial de la homografía
    pl = positions_plausibility(df_pos)
    print('Homografía:', '🟢 OK' if pl['ok'] else '🔴 REVISAR')
    print(f"  Dentro de la cancha: {pl['pct_dentro_cancha']}%")
    print(f"  Rango X: {pl['x_rango']} | Rango Y: {pl['y_rango']}")

    # Jugadores por frame (esperable ~10-11 por equipo; ojo con expulsiones —
    # ver sofascore/incidents_clean.csv antes de asumir que el tracking falló)
    print('\nJugadores detectados por frame:')
    print(players_per_frame(df_pos).to_string(index=False))

    # Cruce con lineups oficiales
    lineups_path = config.find_match_sofascore(MATCH_KEY, SEASON, 'lineups_clean.csv')
    if lineups_path:
        df_lineups = pd.read_csv(lineups_path)
        result = compare_with_lineups(df_pos, df_lineups)
        if result:
            print('\nOficial Sofascore :', result['oficial_sofascore'])
            print('Tracking del video:', result['tracking_video'])
            if result['alertas']:
                for a in result['alertas']:
                    print(f'⚠️  {a}')
            else:
                print('🟢 Sin alertas — tracking consistente con Sofascore.')
    else:
        print(f'\n⚠️  No hay lineups_clean.csv en {config.match_sofascore_dir(MATCH_KEY, SEASON)}')

    # Resumen de tracks (top 25) — para inspeccionar la fragmentación (F3)
    print('\nTracks principales:')
    print(track_summary(df_pos).head(25).to_string(index=False))
else:
    print('Sin datos de tracking para validar.')

## 4. Vista rápida del tracking

Un instante del partido dibujado en la cancha: cada punto es un jugador en su
posición real según el video. Si esto se ve como una formación de fútbol
coherente, la homografía funciona.

In [ ]:
if df_pos is not None and not df_pos.empty:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from mplsoccer import Pitch

    BG_COLOR, TEXT_COLOR = '#0e1a10', '#e8e9e4'

    # Elegir un segundo del video para inspeccionar
    INSTANTE = float(df_pos['t_sec'].median())
    frame_df = df_pos[df_pos['t_sec'] == df_pos.loc[(df_pos['t_sec'] - INSTANTE).abs().idxmin(), 't_sec']]

    fig, ax = plt.subplots(figsize=(11, 7.5))
    fig.patch.set_facecolor(BG_COLOR)
    pitch = Pitch(pitch_type='opta', pitch_color='#1a3a1a', line_color='white', linewidth=1.2)
    pitch.draw(ax=ax)

    COLORS = {0: '#00d4aa', 1: '#cc3333', -1: '#FFD700'}
    for team_id, group in frame_df.groupby('team'):
        pitch.scatter(group['x'], group['y'], s=250, c=COLORS.get(team_id, 'white'),
                      edgecolor='white', linewidth=1.2, alpha=0.9, ax=ax,
                      label=f'Equipo {team_id}' if team_id >= 0 else 'Árbitro')

    if df_ball is not None and not df_ball.empty:
        b = df_ball.iloc[(df_ball['t_sec'] - INSTANTE).abs().idxmin()]
        pitch.scatter([b['x']], [b['y']], s=120, c='white', edgecolor='black',
                      linewidth=1.5, ax=ax, zorder=5, label='Pelota')

    ax.legend(facecolor=BG_COLOR, labelcolor=TEXT_COLOR, fontsize=9)
    ax.set_title(f'Tracking — minuto {INSTANTE/60:.1f} de partido',
                 color=TEXT_COLOR, fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(os.path.join(TRACKING_DIR, 'vista_tracking.png'),
                dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    print('✅ vista_tracking.png guardada en tracking/')
    plt.close(fig)